In [1]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np
import re

In [2]:
import re
from pathlib import Path

# TABLES_DIR_SP = Path("/data/critt/tprdb/MIGRATION/BML12/Tables")
TABLES_DIR_SP = Path("/data/critt/tprdb/PUBLIC/BML12/Tables")
# TABLES_DIR_GE = Path("/data/critt/tprdb/MIGRATION/SG12/Tables")
TABLES_DIR_GE = Path("/data/critt/tprdb/PUBLIC/SG12/Tables")
# Matches anything ending in T<digits>.pu or T<digits>.fd  (e.g. P02_T3.pu, P14_T12.fd)
pu_pattern = re.compile(r"T\d+\.pu$")
fd_pattern = re.compile(r"T\d+\.fd$")

def load_concat(directory, pattern):
    files = sorted(p for p in directory.iterdir() if pattern.search(p.name))
    if not files:
        raise FileNotFoundError(f"No files matching {pattern.pattern} in {directory}")
    frames = []
    for f in files:
        df = pd.read_csv(f, sep='\t')
        df['SourceFile'] = f.name        # handy for debugging / provenance
        frames.append(df)
    return pd.concat(frames, ignore_index=True)

PU_S = load_concat(TABLES_DIR_SP, pu_pattern)
FD_S = load_concat(TABLES_DIR_SP, fd_pattern)
PU_G = load_concat(TABLES_DIR_GE, pu_pattern)
FD_G = load_concat(TABLES_DIR_GE, fd_pattern)

print(f"PU_S: {len(PU_S)} rows from {PU_S['SourceFile'].nunique()} files")
print(f"FD_S: {len(FD_S)} rows from {FD_S['SourceFile'].nunique()} files")
print("Participants in PU_S:", sorted(PU_S['Part'].unique()))
print("Sessions in PU_S:    ", sorted(PU_S['Session'].unique()))
print(f"PU_G: {len(PU_G)} rows from {PU_G['SourceFile'].nunique()} files")
print(f"FD_G: {len(FD_G)} rows from {FD_G['SourceFile'].nunique()} files")
print("Participants in PU_G:", sorted(PU_G['Part'].unique()))
print("Sessions in PU_G:    ", sorted(PU_G['Session'].unique()))

PU_S: 5278 rows from 60 files
FD_S: 73889 rows from 60 files
Participants in PU_S: ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08', 'P09', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P16', 'P17', 'P18', 'P19', 'P20', 'P21', 'P22', 'P23', 'P24', 'P25', 'P26', 'P27', 'P28', 'P29', 'P30', 'P31', 'P32']
Sessions in PU_S:     ['P01_T1', 'P01_T2', 'P02_T3', 'P02_T4', 'P03_T5', 'P03_T6', 'P04_T1', 'P04_T2', 'P05_T3', 'P05_T4', 'P06_T5', 'P06_T6', 'P07_T1', 'P07_T3', 'P08_T3', 'P08_T5', 'P09_T1', 'P09_T5', 'P10_T2', 'P10_T4', 'P11_T1', 'P11_T3', 'P12_T3', 'P12_T5', 'P13_T1', 'P13_T5', 'P14_T2', 'P14_T4', 'P15_T4', 'P15_T6', 'P16_T2', 'P16_T6', 'P17_T3', 'P17_T6', 'P18_T2', 'P18_T5', 'P19_T4', 'P20_T4', 'P20_T5', 'P21_T1', 'P21_T6', 'P22_T2', 'P22_T3', 'P23_T1', 'P23_T2', 'P24_T4', 'P25_T6', 'P26_T1', 'P26_T2', 'P27_T3', 'P27_T4', 'P28_T5', 'P28_T6', 'P29_T1', 'P29_T5', 'P30_T1', 'P30_T3', 'P31_T3', 'P31_T5', 'P32_T4']
PU_G: 3586 rows from 47 files
FD_G: 199844 rows from 47 files
Partici

In [3]:
# Phase distribution from raw PU tables (pre-filter, includes O/D/R)

def phase_breakdown(PU, label):
    print(f"\n{'=' * 50}")
    print(f"  {label}")
    print('=' * 50)
    print(f"Total PUs: {len(PU)}")

    counts = PU['Phase'].value_counts(dropna=False).sort_index()
    props  = PU['Phase'].value_counts(normalize=True, dropna=False).sort_index()

    print(f"\nPhase counts and proportions:")
    summary = pd.DataFrame({
        'Count':     counts,
        'Proportion': props.round(3),
    })
    print(summary.to_string())

    print(f"\nPer-translator phase counts:")
    pivot = PU.groupby(['Part', 'Phase']).size().unstack(fill_value=0)
    # Add row totals
    pivot['Total'] = pivot.sum(axis=1)
    print(pivot.to_string())

    print(f"\nPer-translator D-phase proportion:")
    d_props = (pivot.drop(columns='Total').get('D', pd.Series(0)) / pivot['Total']).round(3)
    print(d_props.describe().round(3))

phase_breakdown(PU_S, "Spanish (BML12)")
phase_breakdown(PU_G, "German (SG12)")


  Spanish (BML12)
Total PUs: 5278

Phase counts and proportions:
       Count  Proportion
Phase                   
D       4704       0.891
R        574       0.109

Per-translator phase counts:
Phase    D    R  Total
Part                  
P01    167   26    193
P02    127   22    149
P03    142    7    149
P04    181    4    185
P05    157    0    157
P06     91  110    201
P07    154   11    165
P08    112   49    161
P09    143   11    154
P10    154   11    165
P11    182   19    201
P12    156   19    175
P13    154   15    169
P14    188    2    190
P15    116   18    134
P16    233   34    267
P17    163   56    219
P18    170   10    180
P19     73    3     76
P20    164   15    179
P21    158    0    158
P22    186    0    186
P23    182   25    207
P24     55    2     57
P25     68   12     80
P26    185    5    190
P27    148    1    149
P28    140   17    157
P29    164   32    196
P30    151   24    175
P31    182    6    188
P32     58    8     66

Per-translator D-phas

In [4]:
FIX_TIME_COL = 'Time'   

def _split_tgids(s):
    """'1+2+3' -> {'1','2','3'};   NaN/'' -> empty set"""
    if pd.isna(s) or s == '':
        return set()
    return {tok for tok in str(s).split('+') if tok != ''}

def build_trial_table(PU, FD):
    """Reshape PU rows into trial rows and aggregate fixations in each pause."""
    # ------------------------------------------------------------------
    # 1.  Reshape PU rows into trial rows
    pu_sorted = (
        PU.sort_values(['Study', 'Session', 'Part', 'Time'])
          .reset_index(drop=True)
    )

    def _build(group):
        g = group.sort_values('Time').reset_index(drop=True)
        if len(g) < 2:
            return pd.DataFrame()
        prev = g.iloc[:-1].reset_index(drop=True)
        nxt  = g.iloc[1:].reset_index(drop=True)
        return pd.DataFrame({
            'Study':         prev['Study'],
            'Session':       prev['Session'],
            'Part':          prev['Part'],
            'PUB':           prev['PUB'],
            'Prev_PU_Time':  prev['Time'],
            'Prev_PU_End':   prev['End'],
            'Prev_PU_Type':  prev['Type'],
            'Prev_PU_Phase': prev['Phase'],
            'Prev_PU_TGid':  prev['TGid'],
            'Prev_PU_Edit':  prev['Edit'],
            'Pause':         nxt['Pause'],
            'Pause_Start':   prev['End'],
            'Pause_End':     nxt['Time'],
            'Next_PU_Time':  nxt['Time'],
            'Next_PU_End':   nxt['End'],
            'Next_PU_Type':  nxt['Type'],
            'Next_PU_TGid':  nxt['TGid'],
            'Next_PU_Edit':  nxt['Edit'],
        })

    trial_table = (
        pu_sorted.groupby(['Study', 'Session', 'Part'], group_keys=False)
                 .apply(_build)
                 .reset_index(drop=True)
    )

    # ------------------------------------------------------------------
    # 2.  Aggregate fixations inside each pause window
    fd_by_key = {k: v.reset_index(drop=True)
                 for k, v in FD.groupby(['Study', 'Session', 'Part'])}

    def _fix_stats(row):
        key = (row['Study'], row['Session'], row['Part'])
        fd  = fd_by_key.get(key)
        if fd is None or len(fd) == 0:
            return pd.Series({'Num_Fixations': 0,
                              'Fixated_TGids': '',
                              'Relevant_Fixations': 0})

        in_pause = fd[(fd[FIX_TIME_COL] >= row['Pause_Start']) &
                      (fd[FIX_TIME_COL] <= row['Pause_End'])]

        tg_strings = in_pause['TGid'].dropna().astype(str).tolist()

        all_tg = set()
        for s in tg_strings:
            all_tg |= _split_tgids(s)
        try:
            sorted_tg = sorted(all_tg, key=lambda x: int(x))
        except ValueError:
            sorted_tg = sorted(all_tg)
        fixated_str = '+'.join(sorted_tg)

        prev_tg  = _split_tgids(row['Prev_PU_TGid'])
        relevant = sum(1 for s in tg_strings if _split_tgids(s) & prev_tg)

        return pd.Series({'Num_Fixations':      len(in_pause),
                          'Fixated_TGids':      fixated_str,
                          'Relevant_Fixations': relevant})

    fix_cols = trial_table.apply(_fix_stats, axis=1)
    trial_table = pd.concat([trial_table, fix_cols], axis=1)

    # ------------------------------------------------------------------
    # 3.  Trial number per translator
    trial_table = trial_table.sort_values(['Part', 'Pause_Start']).reset_index(drop=True)
    trial_table['Trial_Number'] = trial_table.groupby('Part').cumcount() + 1

    # ------------------------------------------------------------------
    # 4.  Tidy column order
    return trial_table[[
        'Study', 'Session', 'Part', 'Trial_Number', 'PUB',
        'Prev_PU_Time', 'Prev_PU_End', 'Prev_PU_Type',
        'Prev_PU_Phase', 'Prev_PU_TGid', 'Prev_PU_Edit',
        'Pause', 'Pause_Start', 'Pause_End',
        'Num_Fixations', 'Fixated_TGids', 'Relevant_Fixations',
        'Next_PU_Time', 'Next_PU_End', 'Next_PU_Type',
        'Next_PU_TGid', 'Next_PU_Edit',
    ]]

# ------------------------------------------------------------------
# Build both trial tables
trial_table_S = build_trial_table(PU_S, FD_S)
trial_table_G = build_trial_table(PU_G, FD_G)

print(f"Spanish (BML12): {len(trial_table_S)} trials, {trial_table_S['Part'].nunique()} translators")
print(f"German  (SG12):  {len(trial_table_G)} trials, {trial_table_G['Part'].nunique()} translators")

trial_table_S.head(5)

/tmp/ipykernel_2780990/3667734495.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_build)
/tmp/ipykernel_2780990/3667734495.py:48: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_build)


Spanish (BML12): 5218 trials, 32 translators
German  (SG12):  3539 trials, 24 translators


,Study,Session,Part,Trial_Number,PUB,Prev_PU_Time,Prev_PU_End,Prev_PU_Type,Prev_PU_Phase,Prev_PU_TGid,Prev_PU_Edit,Pause,Pause_Start,Pause_End,Num_Fixations,Fixated_TGids,Relevant_Fixations,Next_PU_Time,Next_PU_End,Next_PU_Type,Next_PU_TGid,Next_PU_Edit
0,BML12,P01_T1,P01,1,686,92016,97016,C,D,1+2+3,El_enfere[e]mero_asesiono_re[er_ono,937,97016,97953,0,,0,97953,99266,I,3+4,no_recibe
1,BML12,P01_T1,P01,2,686,97953,99266,I,D,3+4,no_recibe,1140,99266,100406,4,4+5,3,100406,101719,I,4+5,_cuatro_
2,BML12,P01_T1,P01,3,686,100406,101719,I,D,4+5,_cuatro_,1875,101719,103594,5,6+7,0,103594,107781,C,5+7,sentencias_de_vida.__[__.]__
3,BML12,P01_T2,P01,4,951,98031,101828,C,D,1+2,lAS[SAl]Las_familiars[#]_,6187,101828,108015,26,0+1+2+4+6+7+14+95+144,2,108015,111406,C,3+4+5+6+7,el_coste_de_la_f[f]vida_
4,BML12,P01_T1,P01,5,686,103594,107781,C,D,5+7,sentencias_de_vida.__[__.]__,13735,107781,121516,50,1+2+8+9+10+11+12+13+27+28+29+31+32+33+34+35+36...,0,121516,121516,I,8,E


In [5]:
def sanity_checks(trial_table, label):
    print(f"\n===== {label} =====")
    # (a) Pause duration consistency 
    diff = trial_table['Pause'] - (trial_table['Pause_End'] - trial_table['Pause_Start'])
    print("Pause duration diff:")
    print(diff.describe())
    # (b) Trial counts per translator
    print("\nTrials per translator:")
    print(trial_table.groupby('Part').size().describe())
    # (c) Fixations actually picked up — if mostly zero, clock alignment may be wrong
    print("\nNum_Fixations:")
    print(trial_table['Num_Fixations'].describe())
    print(f"Trials with 0 fixations: {(trial_table['Num_Fixations']==0).mean():.3f}")

sanity_checks(trial_table_S, "Spanish (BML12)")
sanity_checks(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Pause duration diff:
count    5218.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64

Trials per translator:
count     32.000000
mean     163.062500
std       44.204027
min       56.000000
25%      150.750000
50%      170.000000
75%      188.000000
max      265.000000
dtype: float64

Num_Fixations:
count    5218.000000
mean        8.628018
std        24.943667
min         0.000000
25%         0.000000
50%         1.000000
75%         7.000000
max       581.000000
Name: Num_Fixations, dtype: float64
Trials with 0 fixations: 0.401

===== German (SG12) =====
Pause duration diff:
count    3539.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
dtype: float64

Trials per translator:
count     24.000000
mean     147.458333
std       25.615856
min      103.000000
25%      128.500000
50%      141.500000
75%      164.500000
ma

In [6]:
print("In-memory trial_table_S:")
print(f"  Total rows: {len(trial_table_S)}")
print(f"  Phase distribution: {trial_table_S['Prev_PU_Phase'].value_counts().to_dict()}")

In-memory trial_table_S:
  Total rows: 5218
  Phase distribution: {'D': 4695, 'R': 523}


In [7]:
def keep_drafting(trial_table, label):
    trial_table = trial_table[trial_table['Prev_PU_Phase'] == 'D'].reset_index(drop=True)
    trial_table = trial_table.sort_values(['Part', 'Pause_Start']).reset_index(drop=True)
    trial_table['Trial_Number'] = trial_table.groupby('Part').cumcount() + 1
    print(f"\n===== {label} =====")
    print(f"{len(trial_table)} drafting trials across {trial_table['Part'].nunique()} translators")
    print(trial_table.groupby('Part').size().describe())
    return trial_table

trial_table_S = keep_drafting(trial_table_S, "Spanish (BML12)")
trial_table_G = keep_drafting(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
4695 drafting trials across 32 translators
count     32.000000
mean     146.718750
std       41.232022
min       55.000000
25%      136.750000
50%      154.500000
75%      172.750000
max      233.000000
dtype: float64

===== German (SG12) =====
3306 drafting trials across 24 translators
count     24.000000
mean     137.750000
std       28.437957
min       80.000000
25%      125.250000
50%      136.500000
75%      151.250000
max      204.000000
dtype: float64


In [8]:
def _split_tgids(s):
    if pd.isna(s) or s == '':
        return set()
    return {t for t in str(s).split('+') if t}

def add_touches_prev(trial_table, label):
    trial_table = trial_table.copy()
    trial_table['Next_Touches_Prev'] = trial_table.apply(
        lambda r: bool(_split_tgids(r['Prev_PU_TGid']) & _split_tgids(r['Next_PU_TGid'])),
        axis=1
    )
    print(f"\n===== {label} =====")
    print("Word-change rate (next PU touches prev PU's TGids):")
    print(trial_table['Next_Touches_Prev'].mean())
    print("\nPer translator:")
    print(trial_table.groupby('Part')['Next_Touches_Prev'].mean().describe())
    print("\nRelevant_Fixations distribution:")
    print(trial_table['Relevant_Fixations'].describe())
    print("\nTrials with 0 relevant fixations:",
          (trial_table['Relevant_Fixations'] == 0).mean())
    return trial_table

trial_table_S = add_touches_prev(trial_table_S, "Spanish (BML12)")
trial_table_G = add_touches_prev(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Word-change rate (next PU touches prev PU's TGids):
0.5365282215122471

Per translator:
count    32.000000
mean      0.527345
std       0.100492
min       0.356643
25%       0.458484
50%       0.506172
75%       0.613585
max       0.713415
Name: Next_Touches_Prev, dtype: float64

Relevant_Fixations distribution:
count    4695.000000
mean        1.458360
std         4.493038
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max       105.000000
Name: Relevant_Fixations, dtype: float64

Trials with 0 relevant fixations: 0.6762513312034079

===== German (SG12) =====
Word-change rate (next PU touches prev PU's TGids):
0.46823956442831216

Per translator:
count    24.000000
mean      0.461106
std       0.098083
min       0.259843
25%       0.382509
50%       0.474609
75%       0.543801
max       0.616580
Name: Next_Touches_Prev, dtype: float64

Relevant_Fixations distribution:
count    3306.000000
mean        4.227163
std       

In [9]:
def next_pu_type_diagnostics(trial_table, label):
    """Diagnostic only: how often does the next PU correct the previous one?

    Note: this reports the same quantity that is later stored as
    Word_Change_Loose (see the robustness-definitions cell). It is printed
    here for inspection but not assigned to a column, to avoid keeping two
    identically-valued columns in the trial table.
    """
    print(f"\n===== {label} =====")
    print("Next PU type breakdown:")
    print(trial_table['Next_PU_Type'].value_counts(dropna=False))
    print("\nTouches-prev rate by next PU type:")
    print(trial_table.groupby('Next_PU_Type')['Next_Touches_Prev']
                     .agg(['mean', 'count']))

    correction_types = ['D', 'C']
    is_correction = (
        trial_table['Next_PU_Type'].isin(correction_types) &
        trial_table['Next_Touches_Prev']
    )
    print("\nUnfiltered correction rate (= Word_Change_Loose):", is_correction.mean())
    print("\nPer translator:")
    print(is_correction.groupby(trial_table['Part']).mean().describe())
    return trial_table

trial_table_S = next_pu_type_diagnostics(trial_table_S, "Spanish (BML12)")
trial_table_G = next_pu_type_diagnostics(trial_table_G, "German (SG12)")



===== Spanish (BML12) =====
Next PU type breakdown:
Next_PU_Type
I    2885
C    1486
D     324
Name: count, dtype: int64

Touches-prev rate by next PU type:
                  mean  count
Next_PU_Type                 
C             0.612382   1486
D             0.830247    324
I             0.464471   2885

Unfiltered correction rate (= Word_Change_Loose): 0.2511182108626198

Per translator:
count    32.000000
mean      0.241920
std       0.060268
min       0.145455
25%       0.191249
50%       0.239679
75%       0.291395
max       0.377682
dtype: float64

===== German (SG12) =====
Next PU type breakdown:
Next_PU_Type
I    1990
C    1109
D     207
Name: count, dtype: int64

Touches-prev rate by next PU type:
                  mean  count
Next_PU_Type                 
C             0.552750   1109
D             0.710145    207
I             0.395980   1990

Unfiltered correction rate (= Word_Change_Loose): 0.22988505747126436

Per translator:
count    24.000000
mean      0.223970
std   

In [10]:
# Helpers 
LETTER_RE = re.compile(r'[^\W\d_]', re.UNICODE)

def has_substantive_deletion(edit_str):
    """True if the deletion content is substantive (not just a typo)."""
    if pd.isna(edit_str):
        return False
    deletions = re.findall(r'\[([^\]]*)\]', str(edit_str))
    for d in deletions:
        n_letters = len(LETTER_RE.findall(d))
        if '_' in d and n_letters >= 2:   # crosses word boundary w/ real content
            return True
        if n_letters >= 5:                # long within-word rewrite
            return True
    return False

def _split_tgids(s):
    if pd.isna(s) or s == '':
        return set()
    return {t for t in str(s).split('+') if t}

# ------------------------------------------------------------------
# Build Word_Change and 2-bin Task_2 for one trial table
def assign_two_bin_task2(trial_table, label):
    trial_table = trial_table.copy()

    # Step 1: TGid overlap 
    if 'Next_Touches_Prev' not in trial_table.columns:
        trial_table['Next_Touches_Prev'] = trial_table.apply(
            lambda r: bool(_split_tgids(r['Prev_PU_TGid']) & _split_tgids(r['Next_PU_TGid'])),
            axis=1
        )

    # Step 2: Substantive-deletion flag
    trial_table['Next_Has_Substantive_Del'] = trial_table['Next_PU_Edit'].apply(has_substantive_deletion)

    # Step 3: Word_Change = correction in next PU + overlaps prev + substantive
    trial_table['Word_Change'] = (
        trial_table['Next_PU_Type'].isin(['D', 'C']) &
        trial_table['Next_Touches_Prev'] &
        trial_table['Next_Has_Substantive_Del']
    )

    # Step 4: Binary Task 2 — 1 = correction (low confidence), 2 = no correction (high confidence)
    trial_table['Task_2'] = np.where(trial_table['Word_Change'], 1, 2)

    # ------------------------------------------------------------------
    # Sanity checks
    print(f"\n===== {label} =====")
    print(f"Word_Change rate: {trial_table['Word_Change'].mean():.3f}")
    print(f"\nTask_2 distribution:")
    print(trial_table['Task_2'].value_counts().sort_index())
    print("\nProportions:")
    print(trial_table['Task_2'].value_counts(normalize=True).sort_index().round(3))

    print("\nPer-translator bin counts:")
    per_part = trial_table.groupby(['Part', 'Task_2']).size().unstack(fill_value=0)
    print(per_part)
    print(f"\nTranslators with at least one empty bin: {(per_part == 0).any(axis=1).sum()} of {len(per_part)}")

    return trial_table

# ------------------------------------------------------------------
# Apply to both studies
trial_table_S = assign_two_bin_task2(trial_table_S, "Spanish (BML12)")
trial_table_G = assign_two_bin_task2(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Word_Change rate: 0.063

Task_2 distribution:
Task_2
1     298
2    4397
Name: count, dtype: int64

Proportions:
Task_2
1    0.063
2    0.937
Name: proportion, dtype: float64

Per-translator bin counts:
Task_2   1    2
Part           
P01      7  160
P02      6  121
P03      8  134
P04     18  163
P05     12  143
P06     11   80
P07      8  146
P08      6  106
P09      4  139
P10     11  142
P11      8  174
P12      7  149
P13      9  145
P14     15  173
P15      4  112
P16     24  209
P17     10  153
P18     11  159
P19      5   68
P20     13  151
P21      6  150
P22     12  172
P23     11  171
P24      0   55
P25      5   63
P26      9  176
P27     12  135
P28     10  130
P29     14  150
P30      5  146
P31     11  170
P32      6   52

Translators with at least one empty bin: 1 of 32

===== German (SG12) =====
Word_Change rate: 0.059

Task_2 distribution:
Task_2
1     195
2    3111
Name: count, dtype: int64

Proportions:
Task_2
1    0.059
2    0.941
Name:

In [11]:
# Robustness: alternative word-change definitions 

def has_phrasal_deletion(edit_str):
    """Tight criterion: deletion has >=8 letters AND crosses a word boundary."""
    if pd.isna(edit_str):
        return False
    for d in re.findall(r'\[([^\]]*)\]', str(edit_str)):
        n_letters = len(LETTER_RE.findall(d))
        if '_' in d and n_letters >= 8:
            return True
    return False

def add_robustness_definitions(trial_table, label):
    trial_table = trial_table.copy()

    # LOOSE: any correction overlapping prev TGids (no content filter)
    trial_table['Word_Change_Loose'] = (
        trial_table['Next_PU_Type'].isin(['D', 'C']) &
        trial_table['Next_Touches_Prev']
    )

    # TIGHT: requires a clearly phrasal deletion
    trial_table['Next_Has_Phrasal_Del'] = trial_table['Next_PU_Edit'].apply(has_phrasal_deletion)
    trial_table['Word_Change_Tight'] = (
        trial_table['Next_PU_Type'].isin(['D', 'C']) &
        trial_table['Next_Touches_Prev'] &
        trial_table['Next_Has_Phrasal_Del']
    )

    # 2-bin Task_2 under each definition (1 = correction, 2 = no correction)
    trial_table['Task_2_Loose'] = np.where(trial_table['Word_Change_Loose'], 1, 2)
    trial_table['Task_2_Tight'] = np.where(trial_table['Word_Change_Tight'], 1, 2)

    # ------------------------------------------------------------------
    # Diagnostics
    print(f"\n===== {label} =====")
    print("Word change rates:")
    print(f"  Loose:  {trial_table['Word_Change_Loose'].mean():.3f}")
    print(f"  Medium: {trial_table['Word_Change'].mean():.3f}")
    print(f"  Tight:  {trial_table['Word_Change_Tight'].mean():.3f}")

    print("\nBin distributions (Bin 1 = correction / Bin 2 = no correction):")
    for name, col in [('Loose', 'Task_2_Loose'), ('Medium', 'Task_2'), ('Tight', 'Task_2_Tight')]:
        counts = trial_table[col].value_counts().sort_index()
        print(f"  {name:7}: {counts.to_dict()}")

    print("\nPer-translator Bin 1 counts:")
    bin1_counts = pd.DataFrame({
        'Loose':  trial_table.groupby('Part').apply(lambda g: (g['Task_2_Loose']==1).sum()),
        'Medium': trial_table.groupby('Part').apply(lambda g: (g['Task_2']==1).sum()),
        'Tight':  trial_table.groupby('Part').apply(lambda g: (g['Task_2_Tight']==1).sum()),
    })
    print(bin1_counts)

    print("\nEmpty Bin 1 cells by definition:")
    for name, col in [('Loose', 'Task_2_Loose'), ('Medium', 'Task_2'), ('Tight', 'Task_2_Tight')]:
        per_part = trial_table.groupby('Part').apply(lambda g: (g[col]==1).sum())
        empty = (per_part == 0).sum()
        print(f"  {name:7}: {empty} / {len(per_part)} translators with zero Bin 1 trials")

    return trial_table

# ------------------------------------------------------------------
# Apply to both studies
trial_table_S = add_robustness_definitions(trial_table_S, "Spanish (BML12)")
trial_table_G = add_robustness_definitions(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Word change rates:
  Loose:  0.251
  Medium: 0.063
  Tight:  0.022

Bin distributions (Bin 1 = correction / Bin 2 = no correction):
  Loose  : {1: 1179, 2: 3516}
  Medium : {1: 298, 2: 4397}
  Tight  : {1: 102, 2: 4593}

Per-translator Bin 1 counts:
      Loose  Medium  Tight
Part                      
P01      28       7      0
P02      31       6      3
P03      28       8      0
P04      56      18      5
P05      30      12      7
P06      28      11      2
P07      28       8      2
P08      19       6      2
P09      27       4      0
P10      34      11      4
P11      43       8      2
P12      29       7      3
P13      40       9      2
P14      53      15      4
P15      19       4      1
P16      88      24      8
P17      32      10      5
P18      53      11      6
P19      14       5      2
P20      52      13      7
P21      43       6      3
P22      59      12      2
P23      48      11      5
P24       8       0      0
P25      14       5

/tmp/ipykernel_2780990/1350155295.py:56: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'Loose':  trial_table.groupby('Part').apply(lambda g: (g['Task_2_Loose']==1).sum()),
/tmp/ipykernel_2780990/1350155295.py:57: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  'Medium': trial_table.groupby('Part').apply(lambda g: (g['Task_2']==1).sum()),
/tmp/ipykernel_2780990/1350155295.py:58: DeprecationWarning: DataFrameGr

In [12]:
# Diagnostic: which corrections the substantive-content rule keeps vs demotes

# Word_Change_Loose  = any correction overlapping the previous PU's TGids
# Word_Change        = the same restricted to substantive deletions

def show_kept_vs_demoted(trial_table, label, n=50):
    kept    = trial_table[trial_table['Word_Change']]
    demoted = trial_table[trial_table['Word_Change_Loose'] & ~trial_table['Word_Change']]

    print(f"\n===== {label} =====")
    print(f"Substantive-change rate: {trial_table['Word_Change'].mean():.4f}")
    print(f"{len(kept)} trials kept as substantive change; "
          f"{len(demoted)} demoted (typing repair only)")

    print("\nSample of DEMOTED deletions (typing repair):")
    print(demoted['Next_PU_Edit'].head(n).to_string())
    print("\nSample of KEPT deletions (reformulations):")
    print(kept['Next_PU_Edit'].head(n).to_string())

show_kept_vs_demoted(trial_table_S, "Spanish (BML12)")
show_kept_vs_demoted(trial_table_G, "German (SG12)")



===== Spanish (BML12) =====
Substantive-change rate: 0.0635
298 trials kept as substantive change; 881 demoted (typing repair only)

Sample of DEMOTED deletions (typing repair):
2                           sentencias_de_vida.__[__.]__
5                                                     [e
19                                                  [_ne
22                                                    [_
26                           _la_aliman[na]entación_y_la
33                                                [_ed_y
38                                                  [ed_
41                                                   [rp
45                                          [m]asesinó_a
47                                                   n[d
49                                      _al_dares[se]les
56                      [a]hasta_unos_límites_alarmantes
68                                                    [a
73                            _s[s_]guiran_incrementando
80                     

In [13]:
# UPPER-BOUND Task 2: any correction within the drafting phase
# Lower bound ≤ true metacognitive sensitivity ≤ upper bound

def assign_upper_bound_task2(trial_table, label):
    """Add Task_2_Upper columns by scanning forward through drafting-phase PUs."""
    trial_table = trial_table.copy()

    trial_table = trial_table.sort_values(
        ['Session', 'Part', 'Prev_PU_Time']
    ).reset_index(drop=True)

    sess_groups = {k: g for k, g in trial_table.groupby(['Session', 'Part'])}

    upper_loose  = np.zeros(len(trial_table), dtype=bool)
    upper_medium = np.zeros(len(trial_table), dtype=bool)
    upper_tight  = np.zeros(len(trial_table), dtype=bool)

    for idx, row in trial_table.iterrows():
        orig_tgids = _split_tgids(row['Prev_PU_TGid'])
        if not orig_tgids:
            continue

        # All later trials in the same session
        sess = sess_groups[(row['Session'], row['Part'])]
        later = sess[sess['Prev_PU_Time'] > row['Prev_PU_Time']]

        for _, future_row in later.iterrows():
            # The future trial's Next_PU is what would constitute a correction
            future_next_tgids = _split_tgids(future_row['Next_PU_TGid'])
            if not (orig_tgids & future_next_tgids):
                continue

            future_next_edit = future_row['Next_PU_Edit']
            future_next_type = future_row['Next_PU_Type']
            is_correction_type = future_next_type in ('D', 'C')

            if is_correction_type:
                upper_loose[idx] = True
                if has_substantive_deletion(future_next_edit):
                    upper_medium[idx] = True
                if has_phrasal_deletion(future_next_edit):
                    upper_tight[idx] = True

            if upper_loose[idx] and upper_medium[idx] and upper_tight[idx]:
                break
                
    trial_table['Word_Change_Upper_Loose']  = (
        trial_table['Word_Change_Loose']  | upper_loose
    )
    trial_table['Word_Change_Upper_Medium'] = (
        trial_table['Word_Change']        | upper_medium
    )
    trial_table['Word_Change_Upper_Tight']  = (
        trial_table['Word_Change_Tight']  | upper_tight
    )

    # 2-bin Task_2 under each upper-bound definition
    trial_table['Task_2_Upper_Loose']  = np.where(trial_table['Word_Change_Upper_Loose'],  1, 2)
    trial_table['Task_2_Upper']        = np.where(trial_table['Word_Change_Upper_Medium'], 1, 2)
    trial_table['Task_2_Upper_Tight']  = np.where(trial_table['Word_Change_Upper_Tight'],  1, 2)

    # Diagnostics
    print(f"\n===== {label} =====")
    print("Lower-bound (immediate next PU only) word-change rates:")
    print(f"  Loose:  {trial_table['Word_Change_Loose'].mean():.3f}")
    print(f"  Medium: {trial_table['Word_Change'].mean():.3f}")
    print(f"  Tight:  {trial_table['Word_Change_Tight'].mean():.3f}")
    print("\nUpper-bound (any drafting-phase correction) word-change rates:")
    print(f"  Loose:  {trial_table['Word_Change_Upper_Loose'].mean():.3f}")
    print(f"  Medium: {trial_table['Word_Change_Upper_Medium'].mean():.3f}")
    print(f"  Tight:  {trial_table['Word_Change_Upper_Tight'].mean():.3f}")
    print("\nGain from lower to upper bound (proportion of trials moved to Bin 1):")
    for lo_col, up_col, name in [
        ('Word_Change_Loose',  'Word_Change_Upper_Loose',  'Loose '),
        ('Word_Change',        'Word_Change_Upper_Medium', 'Medium'),
        ('Word_Change_Tight',  'Word_Change_Upper_Tight',  'Tight '),
    ]:
        gain = (trial_table[up_col] & ~trial_table[lo_col]).mean()
        print(f"  {name}: +{gain:.3f}")
    return trial_table

# Apply to both studies
trial_table_S = assign_upper_bound_task2(trial_table_S, "Spanish (BML12)")
trial_table_G = assign_upper_bound_task2(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Lower-bound (immediate next PU only) word-change rates:
  Loose:  0.251
  Medium: 0.063
  Tight:  0.022

Upper-bound (any drafting-phase correction) word-change rates:
  Loose:  0.391
  Medium: 0.153
  Tight:  0.073

Gain from lower to upper bound (proportion of trials moved to Bin 1):
  Loose : +0.140
  Medium: +0.090
  Tight : +0.051

===== German (SG12) =====
Lower-bound (immediate next PU only) word-change rates:
  Loose:  0.230
  Medium: 0.059
  Tight:  0.015

Upper-bound (any drafting-phase correction) word-change rates:
  Loose:  0.397
  Medium: 0.174
  Tight:  0.065

Gain from lower to upper bound (proportion of trials moved to Bin 1):
  Loose : +0.167
  Medium: +0.115
  Tight : +0.050


### Excluding corrective production units from Task 1

A Task 1 trial records an initial production choice - a stretch of target text the
translator committed to for the first time. A production unit that exists in order to
reformulate an earlier unit is a second attempt. These trials are removed. A production unit is corrective if it reformulates any earlier drafting-phase unit in the same session, judged by the same three conditions used for
Task 2: it is a deletion or combined edit, its target tokens overlap those of the earlier
unit, and the deleted material is substantive.


In [14]:
# Drop trials whose own production unit was itself a reformulation
# run assign_upper_bound_task2 
# A production unit is corrective if it reformulates ANY earlier drafting PU

def corrective_pu_keys(trial_table):
    """(Session, Part, Time) of every PU that reformulates an earlier PU."""
    keys = set()
    for (sess, part), g in trial_table.groupby(['Session', 'Part']):
        g = g.sort_values('Prev_PU_Time')
        prev_times = g['Prev_PU_Time'].values
        prev_tgids = [_split_tgids(x) for x in g['Prev_PU_TGid']]

        for _, u in g.iterrows():
            # Candidate corrective PU = the Next_PU of this trial
            if u['Next_PU_Type'] not in ('D', 'C'):
                continue
            if not has_substantive_deletion(u['Next_PU_Edit']):
                continue
            next_tg = _split_tgids(u['Next_PU_TGid'])
            if not next_tg:
                continue
            # Does it touch the tokens of any PU produced before it
            for t_earlier, tg_earlier in zip(prev_times, prev_tgids):
                if t_earlier < u['Next_PU_Time'] and (next_tg & tg_earlier):
                    keys.add((sess, part, u['Next_PU_Time']))
                    break
    return keys


def drop_corrective_productions(trial_table, label):
    trial_table = trial_table.copy()

    keys = corrective_pu_keys(trial_table)
    trial_table['PrevPU_Was_Reformulation'] = [
        (r['Session'], r['Part'], r['Prev_PU_Time']) in keys
        for _, r in trial_table.iterrows()
    ]

    n_before = len(trial_table)
    kept = trial_table[~trial_table['PrevPU_Was_Reformulation']].reset_index(drop=True)
    kept['Trial_Number'] = kept.groupby('Part').cumcount() + 1
    n_dropped = n_before - len(kept)

    print(f"\n===== {label} =====")
    print(f"Trials before: {n_before}")
    print(f"  dropped (PU reformulated an earlier PU): {n_dropped} ({n_dropped / n_before:.1%})")
    print(f"Trials after:  {len(kept)}")
    print(f"\nReformulation rates after exclusion:")
    print(f"  lower bound: {(kept['Task_2'] == 1).mean():.4f}")
    print(f"  upper bound: {(kept['Task_2_Upper'] == 1).mean():.4f}")
    sizes = kept.groupby('Part').size()
    print(f"Trials per translator: {sizes.min()}-{sizes.max()} (mean {sizes.mean():.1f})")
    return kept


trial_table_S = drop_corrective_productions(trial_table_S, "Spanish (BML12)")
trial_table_G = drop_corrective_productions(trial_table_G, "German (SG12)")



===== Spanish (BML12) =====
Trials before: 4695
  dropped (PU reformulated an earlier PU): 330 (7.0%)
Trials after:  4365

Reformulation rates after exclusion:
  lower bound: 0.0619
  upper bound: 0.1503
Trials per translator: 52-209 (mean 136.4)

===== German (SG12) =====
Trials before: 3306
  dropped (PU reformulated an earlier PU): 258 (7.8%)
Trials after:  3048

Reformulation rates after exclusion:
  lower bound: 0.0584
  upper bound: 0.1713
Trials per translator: 73-188 (mean 127.0)


In [15]:
for label, tt in [('BML12', trial_table_S), ('SG12', trial_table_G)]:
    p = tt['Pause'].dropna()
    q1, med, q3 = p.quantile([.25, .5, .75])
    print(f"{label}: median={med:.0f} IQR({q1:.0f}-{q3:.0f}) min={p.min():.0f} <1000ms={(p<1000).mean()*100:.1f}%")

BML12: median=1891 IQR(1094-3797) min=437 <1000ms=20.0%
SG12: median=3604 IQR(1997-8132) min=624 <1000ms=4.7%


In [16]:
from difflib import SequenceMatcher

def partial_ratio_stdlib(needle, haystack):
    """
    Approximate rapidfuzz.partial_ratio using stdlib only.
    Slides the shorter string over the longer one and returns the
    best-aligning similarity, scaled 0-100.
    """
    if not needle or not haystack:
        return 0
    if len(needle) > len(haystack):
        needle, haystack = haystack, needle
    n = len(needle)
    best = 0.0
    # Slide the needle across the haystack, checking similarity at each position
    for start in range(len(haystack) - n + 1):
        window = haystack[start:start + n]
        ratio = SequenceMatcher(None, needle, window).ratio()
        if ratio > best:
            best = ratio
            if best == 1.0:
                break
    return int(round(best * 100))

In [17]:
import pandas as pd
import re
from pathlib import Path
from difflib import SequenceMatcher

# ------------------------------------------------------------------
# Per-study TABLES_DIR 
TABLES_DIR_S = Path("/data/critt/tprdb/PUBLIC/BML12/Tables")   # Spanish
TABLES_DIR_G = Path("/data/critt/tprdb/PUBLIC/SG12/Tables")    # German
# ------------------------------------------------------------------
# Helpers
def strip_brackets(edit_str):
    """Apply CRITT edit notation: each [xyz] deletes the chars typed just before."""
    if not isinstance(edit_str, str):
        return ""
    out = []
    i = 0
    while i < len(edit_str):
        ch = edit_str[i]
        if ch == '[':
            j = edit_str.find(']', i + 1)
            if j == -1:
                out.append(edit_str[i:])
                break
            deletion = edit_str[i+1:j]
            for _ in range(len(deletion)):
                if out:
                    out.pop()
            i = j + 1
        else:
            out.append(ch)
            i += 1
    return ''.join(out)

def normalize_for_match(s):
    if not isinstance(s, str):
        return ""
    s = s.replace('_', ' ').lower()
    return re.sub(r'\s+', ' ', s).strip()

def partial_ratio_stdlib(needle, haystack):
    if not needle or not haystack:
        return 0
    if len(needle) > len(haystack):
        needle, haystack = haystack, needle
    n = len(needle)
    best = 0.0
    for start in range(len(haystack) - n + 1):
        ratio = SequenceMatcher(None, needle, haystack[start:start + n]).ratio()
        if ratio > best:
            best = ratio
            if best == 1.0:
                break
    return int(round(best * 100))

THRESHOLD     = 75
MIN_NET_CHARS = 2

# ------------------------------------------------------------------
# Per-study Task 1 grader
def add_task1(trial_table, tables_dir, label):
    """Load .tt lookups for this study's sessions and grade every trial."""
    trial_table = trial_table.copy()
    tables_dir  = Path(tables_dir)

    # 1. Load .tt files for the sessions in this study
    def load_tt(session):
        path = tables_dir / f"{session}.tt"
        if not path.exists():
            return None
        return pd.read_csv(path, sep='\t')

    sessions = trial_table['Session'].unique()
    tt_by_session = {s: load_tt(s) for s in sessions}
    missing = [s for s, df in tt_by_session.items() if df is None]
    if missing:
        print(f"  WARNING ({label}): missing .tt for {missing}")
    tgid_lookup = {
        s: dict(zip(df['TTid'], df['TToken']))
        for s, df in tt_by_session.items() if df is not None
    }
    print(f"\n===== {label} =====")
    print(f"Loaded .tt lookups for {len(tgid_lookup)} sessions")

    # 2. Resolve TGid -> final-translation span (closure over tgid_lookup)
    def tgids_to_final_span(tgid_str, session):
        if pd.isna(tgid_str) or not isinstance(tgid_str, str):
            return ""
        lookup = tgid_lookup.get(session)
        if lookup is None:
            return ""
        tokens = []
        for tid_str in tgid_str.split('+'):
            try:
                tid = int(tid_str)
            except ValueError:
                continue
            tok = lookup.get(tid)
            if tok is not None:
                tokens.append(str(tok))
        return ' '.join(tokens)

    # 3. Task 1 scorers
    def task1_correct(edit, tgid, session):
        span = tgids_to_final_span(tgid, session)
        if not span:
            return None
        net = normalize_for_match(strip_brackets(edit))
        if len(net) < MIN_NET_CHARS:
            return None
        return int(partial_ratio_stdlib(net, normalize_for_match(span)) >= THRESHOLD)

    def task1_score(edit, tgid, session):
        span = tgids_to_final_span(tgid, session)
        if not span:
            return None
        net = normalize_for_match(strip_brackets(edit))
        if len(net) < MIN_NET_CHARS:
            return None
        return partial_ratio_stdlib(net, normalize_for_match(span))

    # 4. Apply to the trial table
    trial_table['Final_Span'] = trial_table.apply(
        lambda r: tgids_to_final_span(r['Prev_PU_TGid'], r['Session']), axis=1)
    trial_table['Task_1'] = trial_table.apply(
        lambda r: task1_correct(r['Prev_PU_Edit'], r['Prev_PU_TGid'], r['Session']), axis=1)
    trial_table['Task_1_Score'] = trial_table.apply(
        lambda r: task1_score(r['Prev_PU_Edit'], r['Prev_PU_TGid'], r['Session']), axis=1)

    # 5. Diagnostics
    print(f"Trials evaluated: {trial_table['Task_1'].notna().sum()}")
    print(f"Trials skipped:   {trial_table['Task_1'].isna().sum()}")
    print(f"Proportion correct: {trial_table['Task_1'].mean():.3f}")
    print(f"\nPer-translator accuracy:")
    print(trial_table.groupby('Part')['Task_1'].agg(['mean', 'count']).round(3))
    print(f"\nFuzzy score distribution:")
    print(trial_table['Task_1_Score'].describe().round(1))
    print(f"\nTask_1 x Task_2 cross-tab:")
    print(pd.crosstab(trial_table['Task_1'], trial_table['Task_2'], margins=True))

    print(f"\nSample of Bin 1 trials (corrections; should skew toward Task_1 = 0):")
    bin1 = trial_table[trial_table['Task_2'] == 1][
        ['Prev_PU_Edit', 'Prev_PU_TGid', 'Final_Span', 'Task_1', 'Task_1_Score']
    ].head(10)
    print(bin1.to_string())

    return trial_table


trial_table_S = add_task1(trial_table_S, TABLES_DIR_S, "Spanish (BML12)")
trial_table_G = add_task1(trial_table_G, TABLES_DIR_G, "German (SG12)")


===== Spanish (BML12) =====
Loaded .tt lookups for 60 sessions
Trials evaluated: 3629
Trials skipped:   736
Proportion correct: 0.750

Per-translator accuracy:
       mean  count
Part              
P01   0.724    127
P02   0.784    102
P03   0.838    117
P04   0.759    137
P05   0.750    112
P06   0.714     70
P07   0.824    131
P08   0.763     93
P09   0.873    118
P10   0.761    117
P11   0.671    140
P12   0.737     99
P13   0.761    117
P14   0.750    148
P15   0.835     97
P16   0.625    176
P17   0.680    125
P18   0.735    136
P19   0.720     50
P20   0.556    124
P21   0.740    131
P22   0.826    144
P23   0.700    140
P24   0.950     40
P25   0.769     52
P26   0.811    143
P27   0.784    111
P28   0.844    109
P29   0.593    123
P30   0.849    119
P31   0.715    137
P32   0.795     44

Fuzzy score distribution:
count    3629.0
mean       80.7
std        32.0
min         0.0
25%        74.0
50%       100.0
75%       100.0
max       100.0
Name: Task_1_Score, dtype: float64

Ta

In [18]:
import string

def is_trivial_span(s):
    if not isinstance(s, str) or not s.strip():
        return True
    return all(ch in string.punctuation + string.whitespace for ch in s)

def skipped_diagnostics(trial_table, label):
    print(f"\n===== {label} =====")
    skipped = trial_table[trial_table['Task_1'].isna()]
    print(f"Total skipped: {len(skipped)}")
    print(f"  Trivial Final_Span: {skipped['Final_Span'].apply(is_trivial_span).sum()}")
    short_net = skipped['Prev_PU_Edit'].apply(
        lambda e: len(normalize_for_match(strip_brackets(str(e)) if pd.notna(e) else "")) < MIN_NET_CHARS
    )
    print(f"  Short net content (<{MIN_NET_CHARS} chars): {short_net.sum()}")

    print("\nDropped by bin:")
    print(skipped['Task_2'].value_counts().sort_index())
    print("\nDrop rate by bin:")
    for bin_val in [1, 2]:                       # 2-bin scheme: 1 = correction, 2 = no correction
        total = (trial_table['Task_2'] == bin_val).sum()
        dropped_n = (skipped['Task_2'] == bin_val).sum()
        if total:
            print(f"  Bin {bin_val}: {dropped_n}/{total} = {dropped_n/total:.1%}")
        else:
            print(f"  Bin {bin_val}: 0/0")

skipped_diagnostics(trial_table_S, "Spanish (BML12)")
skipped_diagnostics(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Total skipped: 736
  Trivial Final_Span: 188
  Short net content (<2 chars): 736

Dropped by bin:
Task_2
1     31
2    705
Name: count, dtype: int64

Drop rate by bin:
  Bin 1: 31/270 = 11.5%
  Bin 2: 705/4095 = 17.2%

===== German (SG12) =====
Total skipped: 513
  Trivial Final_Span: 178
  Short net content (<2 chars): 513

Dropped by bin:
Task_2
1     12
2    501
Name: count, dtype: int64

Drop rate by bin:
  Bin 1: 12/178 = 6.7%
  Bin 2: 501/2870 = 17.5%


In [19]:
def exclude_trivial_spans(trial_table, label):
    trial_table = trial_table.copy()
    trivial_mask = trial_table['Final_Span'].apply(is_trivial_span)
    trial_table.loc[trivial_mask, 'Task_1']       = pd.NA
    trial_table.loc[trivial_mask, 'Task_1_Score'] = pd.NA

    print(f"\n===== {label} =====")
    print(f"Trials with trivial spans set to NaN: {trivial_mask.sum()}")
    print(f"\nUpdated counts:")
    print(f"  Evaluated:  {trial_table['Task_1'].notna().sum()}")
    print(f"  Excluded:   {trial_table['Task_1'].isna().sum()}")
    print(f"  Proportion correct: {trial_table['Task_1'].mean():.3f}")
    print("\nUpdated Task_1 x Task_2 cross-tab:")
    print(pd.crosstab(trial_table['Task_1'], trial_table['Task_2'], margins=True))
    print("\nPer-translator accuracy:")
    print(trial_table.groupby('Part')['Task_1'].agg(['mean', 'count']).round(3))
    return trial_table

trial_table_S = exclude_trivial_spans(trial_table_S, "Spanish (BML12)")
trial_table_G = exclude_trivial_spans(trial_table_G, "German (SG12)")


===== Spanish (BML12) =====
Trials with trivial spans set to NaN: 317

Updated counts:
  Evaluated:  3500
  Excluded:   865
  Proportion correct: 0.773

Updated Task_1 x Task_2 cross-tab:
Task_2    1     2   All
Task_1                 
0.0     132   663   795
1.0      85  2620  2705
All     217  3283  3500

Per-translator accuracy:
       mean  count
Part              
P01   0.756    119
P02   0.833     96
P03   0.860    114
P04   0.798    129
P05   0.769    108
P06   0.725     69
P07   0.831    130
P08   0.761     92
P09   0.880    117
P10   0.793    111
P11   0.689    135
P12   0.760     96
P13   0.781    114
P14   0.775    142
P15   0.835     97
P16   0.632    174
P17   0.702    121
P18   0.758    132
P19   0.714     49
P20   0.575    120
P21   0.762    126
P22   0.838    142
P23   0.764    127
P24   0.950     40
P25   0.769     52
P26   0.879    132
P27   0.782    110
P28   0.868    106
P29   0.632    114
P30   0.876    113
P31   0.752    129
P32   0.795     44

===== German (SG12

In [20]:
trial_table_S

,Study,Session,Part,Trial_Number,PUB,Prev_PU_Time,Prev_PU_End,Prev_PU_Type,Prev_PU_Phase,Prev_PU_TGid,Prev_PU_Edit,Pause,Pause_Start,Pause_End,Num_Fixations,Fixated_TGids,Relevant_Fixations,Next_PU_Time,Next_PU_End,Next_PU_Type,Next_PU_TGid,Next_PU_Edit,Next_Touches_Prev,Next_Has_Substantive_Del,Word_Change,Task_2,Word_Change_Loose,Next_Has_Phrasal_Del,Word_Change_Tight,Task_2_Loose,Task_2_Tight,Word_Change_Upper_Loose,Word_Change_Upper_Medium,Word_Change_Upper_Tight,Task_2_Upper_Loose,Task_2_Upper,Task_2_Upper_Tight,PrevPU_Was_Reformulation,Final_Span,Task_1,Task_1_Score
0,BML12,P01_T1,P01,1,686,92016,97016,C,D,1+2+3,El_enfere[e]mero_asesiono_re[er_ono,937,97016,97953,0,,0,97953,99266,I,3+4,no_recibe,True,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,El enfermero asesino,1.0,95.0
1,BML12,P01_T1,P01,2,686,97953,99266,I,D,3+4,no_recibe,1140,99266,100406,4,4+5,3,100406,101719,I,4+5,_cuatro_,True,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,asesino recibe,1.0,100.0
2,BML12,P01_T1,P01,3,686,100406,101719,I,D,4+5,_cuatro_,1875,101719,103594,5,6+7,0,103594,107781,C,5+7,sentencias_de_vida.__[__.]__,True,False,False,2,True,False,False,1,2,True,False,False,1,2,2,False,recibe cuatro,1.0,100.0
3,BML12,P01_T1,P01,4,686,103594,107781,C,D,5+7,sentencias_de_vida.__[__.]__,13735,107781,121516,50,1+2+8+9+10+11+12+13+27+28+29+31+32+33+34+35+36...,0,121516,121516,I,8,E,False,False,False,2,False,False,False,2,2,True,False,False,1,2,2,False,cuatro perpetuas,0.0,19.0
4,BML12,P01_T1,P01,5,686,121516,121516,I,D,8,E,1062,121516,122578,9,8+9+10+11+12+13+14,2,122578,123188,I,8,l_,True,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,El,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4360,BML12,P32_T4,P32,48,1169,370890,371172,C,D,107,"[_,]_",6468,371172,377640,0,,0,377640,385937,C,108+109+110+111+112+113+114+115,deben_adabp[pb]patarse_a_los_efectos_del_cambi...,False,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,desarrollo,NaN,NaN
4361,BML12,P32_T4,P32,49,1169,377640,385937,C,D,108+109+110+111+112+113+114+115,deben_adabp[pb]patarse_a_los_efectos_del_cambi...,1250,385937,387187,0,,0,387187,387265,I,116,._,False,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,deben adaptarse a los efectos del cambio climá...,1.0,98.0
4362,BML12,P32_T4,P32,50,1169,387187,387265,I,D,116,._,6157,387265,393422,0,,0,393422,393843,I,120,La,False,False,False,2,False,False,False,2,2,False,False,False,2,2,2,False,.,NaN,NaN
4363,BML12,P32_T4,P32,51,1169,393422,393843,I,D,120,La,1266,393843,395109,0,,0,395109,406359,C,120+121+122+123+124+125+126+127,os_[_soa]os_esfu[uf]fuerzos_de_adaptación_y_mi...,True,True,True,1,True,False,False,1,2,True,True,False,1,1,2,False,los,0.0,50.0


In [21]:
trial_table_S.to_csv('BML12_2bin_trials.csv', index=False)
trial_table_G.to_csv('SG12_2bin_trials.csv', index=False)
print(f"Saved BML12_2bin_trials.csv: {len(trial_table_S)} rows "
      f"({trial_table_S['Task_1'].notna().sum()} evaluable)")
print(f"Saved SG12_2bin_trials.csv:  {len(trial_table_G)} rows "
      f"({trial_table_G['Task_1'].notna().sum()} evaluable)")

Saved BML12_2bin_trials.csv: 4365 rows (3500 evaluable)
Saved SG12_2bin_trials.csv:  3048 rows (2449 evaluable)


In [22]:
# For each Prev_PU_Phase='D' trial, what phase is the Next_PU
# -do drafting-phase trials have Next_PU with substantive content

# Phase of each PU is in the original PU table

for label, trial_table, pu_table in [('BML12', trial_table_S, PU_S),
                                       ('SG12',  trial_table_G, PU_G)]:
    # Build (Session, Part, Time) -> Phase lookup from PU
    phase_lookup = dict(zip(
        zip(pu_table['Session'], pu_table['Part'], pu_table['Time']),
        pu_table['Phase']
    ))
    next_phases = trial_table.apply(
        lambda r: phase_lookup.get((r['Session'], r['Part'], r['Next_PU_Time']), 'unknown'),
        axis=1
    )
    print(f"\n{label}: Next_PU_Phase distribution for drafting-phase trials")
    print(next_phases.value_counts(normalize=True).round(3))


BML12: Next_PU_Phase distribution for drafting-phase trials
D    0.989
R    0.011
Name: proportion, dtype: float64

SG12: Next_PU_Phase distribution for drafting-phase trials
D    0.988
R    0.012
Name: proportion, dtype: float64


The drafting and revision phases are well-separated in both corpora: 98.9% of next-PUs following a drafting trial are themselves in drafting phase (BML12), 98.8% in SG12. Drafting-phase trials therefore reliably belong to an extended sequential-production episode, and the bound-based analyses are not meaningfully contaminated by interleaved revision activity.